# Telemetry Analysis

Analyse des données de télémétrie machines. Chargement du CSV, puis génération des distributions par machine et par métrique.

## Chargement des données

Import des librairies et lecture du fichier `telemetry.csv`. Les colonnes numériques sont castées en float et `METRICS` / `machines` sont définis ici pour être réutilisés dans les cellules suivantes.

## Valeurs manquantes par machine

Pour chaque métrique, affiche le nombre de valeurs nulles par machine sous forme de graphique en barres groupées.

In [19]:
MISSING_DIR = Path("artifacts/telemetry/missing_values")
MISSING_DIR.mkdir(parents=True, exist_ok=True)

null_counts = (
    df.groupby("machine_id")[[col for col in METRICS.values()]]
    .apply(lambda g: g.isnull().sum())
    .rename(columns={v: k for k, v in METRICS.items()})
)

# Graph global : barres groupées par métrique pour chaque machine
fig, ax = plt.subplots(figsize=(14, 6))
null_counts.plot(kind="bar", ax=ax)
ax.set_title("Valeurs manquantes par machine et par métrique")
ax.set_xlabel("Machine")
ax.set_ylabel("Nombre de valeurs manquantes")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend(title="Métrique")
plt.tight_layout()
fig.savefig(MISSING_DIR / "missing_values_global.png", dpi=100)
plt.close(fig)

# Un graph par métrique
for metric_name in METRICS:
    fig, ax = plt.subplots(figsize=(12, 5))
    null_counts[metric_name].plot(kind="bar", ax=ax, color="darkorange")
    ax.set_title(f"Valeurs manquantes par machine — {metric_name}")
    ax.set_xlabel("Machine")
    ax.set_ylabel("Nombre de valeurs manquantes")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    fig.savefig(MISSING_DIR / f"{metric_name}_missing_values.png", dpi=100)
    plt.close(fig)

print(f"Done. {1 + len(METRICS)} graphs saved to {MISSING_DIR}")

Done. 6 graphs saved to artifacts/telemetry/missing_values


In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

df = pd.read_csv("artifacts/updated_dataset/telemetry.csv")
df = df[df["machine_id"] != "machine_id"]  # drop duplicate headers

for col in ["temperature_c", "pressure_bar", "voltage_mean_v", "rotation_mean_rpm", "pieces_produced"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

METRICS = {
    "temperature": "temperature_c",
    "pressure": "pressure_bar",
    "voltage": "voltage_mean_v",
    "rotation_mean": "rotation_mean_rpm",
    "pieces": "pieces_produced",
}

machines = sorted(df["machine_id"].unique())
print(f"Shape: {df.shape} | Machines: {len(machines)}")
df.head()

Shape: (135626, 7) | Machines: 15


,machine_id,timestamp,temperature_c,pressure_bar,voltage_mean_v,rotation_mean_rpm,pieces_produced
0,MACH-01,2025-06-01 00:00:00,46.348,198.203,227.568,1541.787,4
1,MACH-01,2025-06-01 00:00:00,46.332,198.206,227.570,1541.760,4
2,MACH-01,2025-06-01 01:00:00,48.762,198.295,227.480,1537.860,4
3,MACH-01,2025-06-01 02:00:00,51.352,199.545,228.680,1584.660,13
4,MACH-01,2025-06-01 03:00:00,49.512,201.641,228.440,1588.960,10


## Distribution globale par métrique

Distribution de chaque métrique sur l'ensemble des machines, sans distinction. Les graphiques sont exportés dans `artifacts/telemetry/machines/metrics/`.

In [11]:
METRICS_OUTPUT_DIR = Path("artifacts/telemetry/metrics")
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for metric_name, col in METRICS.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(df[col].dropna().astype(float), kde=True, ax=ax, color="darkorange")
    ax.set_title(f"{metric_name} — global distribution (all machines)")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    plt.tight_layout()
    filename = f"{metric_name}_DISTRIBUTION.png"
    fig.savefig(METRICS_OUTPUT_DIR / filename, dpi=100)
    plt.close(fig)

print(f"Done. {len(METRICS)} graphs saved to {METRICS_OUTPUT_DIR}")

Done. 5 graphs saved to artifacts/telemetry/metrics
